#  Arabic Poetry Generator — GPT-2 Fine-Tuning
### Fine-tune `aragpt2-base` on classical Arabic poetry (Al-Mutanabbi, Ibn Arabi)


1.  Downloads 180k+ classical Arabic poems from HuggingFace
2.  Fine-tunes `aragpt2-base` using HuggingFace Trainer
3.  Evaluates with perplexity
4.  Generates poetry from your own prompts

> **Runtime:** Use `Runtime → Change runtime type → T4 GPU` before running.


## 1.  Install Dependencies

In [1]:
# Install required packages
!pip install -q transformers datasets evaluate accelerate sentencepiece

# Verify GPU
import torch
print(f"PyTorch version : {torch.__version__}")
print(f"CUDA available  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU             : {torch.cuda.get_device_name(0)}")
    print(f"VRAM            : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.1 MB/s eta 0:00:00
PyTorch version : 2.10.0+cu128
CUDA available  : True
GPU             : Tesla T4
VRAM            : 15.6 GB


## 2.  Configuration
Adjust these settings before training. Defaults work well on Colab T4 (15GB VRAM).


In [2]:
# ── Training Configuration ────────────────────────────────────────────────────

# Model
MODEL_NAME   = "aubmindlab/aragpt2-base"  # Arabic GPT-2 pre-trained on 77GB Arabic text
OUTPUT_DIR   = "/content/aragpt2-poetry"  # where to save checkpoints

# Data
MAX_POEMS    = 3000    # number of poems to use (more = better, but slower)
MIN_LINES    = 2       # skip fragments shorter than this
MAX_LINES    = 25      # skip very long poems
BLOCK_SIZE   = 256     # token sequence length per training example

# Training
NUM_EPOCHS   = 10       # 3 epochs is usually enough; increase to 5 for better results
BATCH_SIZE   = 4       # per-device batch size (reduce to 2 if OOM)
GRAD_ACCUM   = 4       # effective batch = BATCH_SIZE × GRAD_ACCUM = 16
LEARNING_RATE = 5e-5
WARMUP_RATIO  = 0.1    # 10% of training steps used for LR warmup

print("Configuration:")
print(f"  Model       : {MODEL_NAME}")
print(f"  Max poems   : {MAX_POEMS:,}")
print(f"  Block size  : {BLOCK_SIZE} tokens")
print(f"  Epochs      : {NUM_EPOCHS}")
print(f"  Eff. batch  : {BATCH_SIZE * GRAD_ACCUM}")
print(f"  LR          : {LEARNING_RATE}")


Configuration:
  Model       : aubmindlab/aragpt2-base
  Max poems   : 3,000
  Block size  : 256 tokens
  Epochs      : 10
  Eff. batch  : 16
  LR          : 5e-05


## 3.  Load & Prepare the Dataset
We use `arbml/ashaar` — 180,000+ classical Arabic poems from HuggingFace.
Poets include Al-Mutanabbi, Ibn Arabi, Al-Buhtiri, Abu Nuwas and more.


In [3]:
import re
import unicodedata
from datasets import load_dataset, Dataset

def clean_arabic(text: str) -> str:
    text = unicodedata.normalize("NFC", text)
    text = re.sub(r'[أإآ]', 'ا', text)
    text = re.sub(r'ة', 'ه', text)
    text = re.sub(r'[ىئ]', 'ي', text)
    text = re.sub(r'[^\u0600-\u06FF\s\n،؟!.]', ' ', text)
    text = re.sub(r'[ \t]+', ' ', text)
    text = re.sub(r'\n{3,}', '\n\n', text)
    return text.strip()

print("Loading arbml/ashaar dataset...")
ds = load_dataset("arbml/ashaar", split="train")
print(f"Total poems in dataset: {len(ds):,}")

text_col = 'poem verses'

poems = []
for row in ds:
    raw = row.get(text_col, []) or []
    if not raw:
        continue
    lines = [clean_arabic(str(l)).strip() for l in raw if str(l).strip()]
    if len(lines) < MIN_LINES:
        continue
    if len(lines) > MAX_LINES:
        lines = lines[:MAX_LINES]
    poems.append('\n'.join(lines))
    if len(poems) >= MAX_POEMS:
        break

print(f"After filtering        : {len(poems):,} poems")
avg_lines = sum(len(p.split('\n')) for p in poems) / len(poems)
print(f"Avg lines per poem     : {avg_lines:.1f}")
print(f"\nSample poem:")
print("─" * 50)
print(poems[0])

Loading arbml/ashaar dataset...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

dataset_infos.json: 0.00B [00:00, ?B/s]

data/train-00000-of-00002.parquet:   0%|          | 0.00/126M [00:00<?, ?B/s]

data/train-00001-of-00002.parquet:   0%|          | 0.00/151M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/254630 [00:00<?, ? examples/s]

Total poems in dataset: 254,630
After filtering        : 3,000 poems
Avg lines per poem     : 14.7

Sample poem:
──────────────────────────────────────────────────
اَصبَحَ المُلك لِلَّذي فَطر الخَل
قَ بِتَقديرٍ للعَزيز العَليمِ
غافر الذَنب للمسيءِ بِعَفوٍ
قابل التَوب ذي العَطاء العَميمِ
مُرسل المُصطَفي البَشير اِلَينا
رَحمه مِنهُ بِالكَلام القَديمِ
رَبَنا رَبّنا اِلَيكَ اَنينا
فَاَجرنا مِن حَر نار الجَحيمِ
وَاكفِنا شَرّ ما نَخاف بِلُطفٍ
يا عَظيماً يَرجي لِكُل عَظيمِ
وَتَقبل اَعمالَنا وَاعفُ عَنا
وَاَنلنا دُخول دار النَعيمِ
بِنَبي بَعثَتهُ فَهَدانا
لِصِراط مِن الهُدي مُستَقيمِ
وَبِمَن نَحنُ في حِماهُ مَدي الدَهر
اَخيهِ يَحيي الحصور الكَريمِ
اَدرك اَدرك قَوماً اَتوا بافتقار
وَاِنكِسار وَمَدمَع مَسجومِ
شَهدت اَرواحَهُم اَنكَ اللَهُ
وَجاءوا بِكُل قَلبٍ سَليم


In [4]:
# Convert to HuggingFace Dataset and split train/eval
raw_dataset = Dataset.from_dict({"text": poems})
split = raw_dataset.train_test_split(test_size=0.05, seed=42)

train_data = split["train"]
eval_data  = split["test"]

print(f"Train poems : {len(train_data):,}")
print(f"Eval  poems : {len(eval_data):,}")


Train poems : 2,850
Eval  poems : 150


## 4.  Load Tokenizer
`aragpt2-base` uses a SentencePiece tokenizer pre-trained on Arabic text.

> **Why not use the English GPT-2 tokenizer?**  
> English GPT-2's vocabulary barely covers Arabic — words break down into single bytes,
> producing extremely long sequences and poor training signal.  
> `aragpt2`'s tokenizer knows Arabic morphology and handles it efficiently.


In [5]:
from transformers import AutoTokenizer

print(f"Loading tokenizer: {MODEL_NAME}")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"Vocab size     : {tokenizer.vocab_size:,}")
print(f"EOS token      : {tokenizer.eos_token!r}  (id={tokenizer.eos_token_id})")
print(f"Max length     : {tokenizer.model_max_length}")

# Demo: show how Arabic text is tokenized
demo_text = "الخيل والليل والبيداء تعرفني"
tokens = tokenizer.tokenize(demo_text)
ids    = tokenizer.encode(demo_text)
print(f"\nDemo tokenization:")
print(f"  Text   : {demo_text!r}")
print(f"  Tokens : {tokens}")
print(f"  IDs    : {ids}")
print(f"  Length : {len(ids)} tokens for {len(demo_text)} chars")


Loading tokenizer: aubmindlab/aragpt2-base


config.json:   0%|          | 0.00/843 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Vocab size     : 64,000
EOS token      : '<|endoftext|>'  (id=0)
Max length     : 1000000000000000019884624838656

Demo tokenization:
  Text   : 'الخيل والليل والبيداء تعرفني'
  Tokens : ['Ø§ÙĦØ®', 'ÙĬÙĦ', 'ĠÙĪØ§ÙĦÙĦÙĬÙĦ', 'ĠÙĪØ§ÙĦØ¨', 'ÙĬØ¯', 'Ø§Ø¡', 'ĠØªØ¹Ø±Ùģ', 'ÙĨÙĬ']
  IDs    : [7636, 517, 59494, 1219, 499, 345, 3407, 495]
  Length : 8 tokens for 28 chars


## 5. Tokenize & Chunk Dataset
We tokenize all poems and chunk them into fixed-length blocks of `BLOCK_SIZE` tokens.

**Why chunking?**  
GPT models need fixed-length inputs. We concatenate all poem tokens (with EOS separators between them)
into one long sequence, then split into chunks. This way no training signal is wasted.


In [6]:
from transformers import DataCollatorForLanguageModeling

def tokenize_and_chunk(dataset, tokenizer, block_size):
    eos_id = tokenizer.eos_token_id

    def tokenize_batch(batch):
        return tokenizer(batch["text"], truncation=False, padding=False)

    def chunk_into_blocks(examples):
        # Concatenate all token ids with EOS separator between poems
        all_ids = []
        for ids in examples["input_ids"]:
            all_ids.extend(ids + [eos_id])

        # Chunk into fixed-length blocks
        total  = (len(all_ids) // block_size) * block_size
        chunks = [all_ids[i:i+block_size] for i in range(0, total, block_size)]

        return {
            "input_ids":      chunks,
            "attention_mask": [[1] * block_size] * len(chunks),
            "labels":         chunks,
        }

    tokenized = dataset.map(tokenize_batch, batched=True,
                            remove_columns=["text"], desc="Tokenizing")
    chunked   = tokenized.map(chunk_into_blocks, batched=True, desc="Chunking")
    return chunked

print("Tokenizing training data...")
train_ds = tokenize_and_chunk(train_data, tokenizer, BLOCK_SIZE)

print("Tokenizing eval data...")
eval_ds  = tokenize_and_chunk(eval_data,  tokenizer, BLOCK_SIZE)

print(f"\nTrain blocks : {len(train_ds):,}  (each = {BLOCK_SIZE} tokens)")
print(f"Eval  blocks : {len(eval_ds):,}")

# DataCollator for Causal LM (mlm=False = GPT-style, NOT masked like BERT)
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)
print("\nDataCollator: mlm=False → causal language modeling (next-token prediction)")


Tokenizing training data...


Tokenizing:   0%|          | 0/2850 [00:00<?, ? examples/s]

Chunking:   0%|          | 0/2850 [00:00<?, ? examples/s]

Tokenizing eval data...


Tokenizing:   0%|          | 0/150 [00:00<?, ? examples/s]

Chunking:   0%|          | 0/150 [00:00<?, ? examples/s]


Train blocks : 3,894  (each = 256 tokens)
Eval  blocks : 223

DataCollator: mlm=False → causal language modeling (next-token prediction)


## 6. Load Model
Load `aragpt2-base` — a 135M parameter Arabic GPT-2.

**Architecture (same as GPT-2 small but trained on Arabic):**
- 12 transformer decoder layers
- 12 attention heads
- 768 embedding dimension  
- 1024 token context window


In [7]:
from transformers import AutoModelForCausalLM

print(f"Loading model: {MODEL_NAME}")
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)

# Move to GPU
device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)

total_params    = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"\nModel loaded on: {device}")
print(f"Total parameters    : {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"\nModel config:")
print(f"  Layers      : {model.config.n_layer}")
print(f"  Heads       : {model.config.n_head}")
print(f"  d_model     : {model.config.n_embd}")
print(f"  Vocab size  : {model.config.vocab_size:,}")
print(f"  Context len : {model.config.n_positions} tokens")


Loading model: aubmindlab/aragpt2-base


model.safetensors:   0%|          | 0.00/553M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: aubmindlab/aragpt2-base
Key                         | Status     |  | 
----------------------------+------------+--+-
h.{0...11}.attn.masked_bias | UNEXPECTED |  | 
h.{0...11}.attn.bias        | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Model loaded on: cuda
Total parameters    : 134,994,432
Trainable parameters: 134,994,432

Model config:
  Layers      : 12
  Heads       : 12
  d_model     : 768
  Vocab size  : 64,000
  Context len : 1024 tokens


## 7.  Train

**Key training decisions:**
- **AdamW optimizer** with `weight_decay=0.01` for regularization
- **LR warmup** (`warmup_ratio=0.1`) — ramps up LR for the first 10% of steps to stabilize early training
- **Cosine LR schedule** — smooth decay after warmup
- **Gradient accumulation** — simulates larger batch without extra memory
- **Early stopping** — stops if eval loss doesn't improve for 2 epochs


In [8]:
import math
from transformers import TrainingArguments, Trainer, EarlyStoppingCallback

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,

    # Epochs & batch
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,

    # Optimizer
    learning_rate=LEARNING_RATE,
    warmup_steps=int(WARMUP_RATIO * NUM_EPOCHS * len(train_ds) // (BATCH_SIZE * GRAD_ACCUM)), # Use warmup_steps instead of warmup_ratio
    weight_decay=0.01,
    lr_scheduler_type="cosine",        # smooth decay after warmup

    # Evaluation & saving
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    save_total_limit=2,                # keep only last 2 checkpoints

    # Speed
    fp16=torch.cuda.is_available(),    # mixed precision on GPU (2× faster)
    dataloader_num_workers=2,

    # Logging
    logging_dir=f"{OUTPUT_DIR}/logs", # Use logging_dir instead of TENSORBOARD_LOGGING_DIR
    logging_steps=50,
    report_to="none",

    push_to_hub=False,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    data_collator=data_collator,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

print("Starting training...")
print(f"  Effective batch size : {BATCH_SIZE * GRAD_ACCUM}")
print(f"  Steps per epoch      : {len(train_ds) // (BATCH_SIZE * GRAD_ACCUM):,}")
print(f"  Total steps          : {len(train_ds) * NUM_EPOCHS // (BATCH_SIZE * GRAD_ACCUM):,}")
print(f"  fp16 (mixed prec.)   : {torch.cuda.is_available()}\n")

train_result = trainer.train()

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


Starting training...
  Effective batch size : 16
  Steps per epoch      : 243
  Total steps          : 2,433
  fp16 (mixed prec.)   : True



`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Epoch,Training Loss,Validation Loss
1,3.591816,3.153245
2,2.957834,2.759327
3,2.778573,2.600619
4,2.523799,2.521819
5,2.489498,2.463046
6,2.407365,2.430140
7,2.348531,2.416047
8,2.401357,2.405241
9,2.383273,2.401277
10,2.319758,2.401625


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['lm_head.weight'].


## 8.  Evaluate

**Perplexity (PP)** measures how well the model predicts the text:
```
PP = exp(cross-entropy loss)
```
- PP = 1 → perfect (model always predicts the exact next token)
- PP = vocab_size → random (model has no idea)
- Fine-tuned Arabic poetry models typically reach PP = 15–40


In [ ]:
import math

eval_results = trainer.evaluate()
ppl = math.exp(eval_results["eval_loss"])

print("=" * 50)
print("  Evaluation Results")
print("=" * 50)
print(f"  Eval loss        : {eval_results['eval_loss']:.4f}")
print(f"  Perplexity (PP)  : {ppl:.2f}")
print(f"  Train loss       : {train_result.training_loss:.4f}")
print()
print("  Perplexity reference:")
print("    PP ~1000  → random model (no learning)")
print("    PP ~50    → decent language model")
print(f"   PP ~{ppl:.0f}   → your fine-tuned model ← you are here")


## 9.  Save Model

In [ ]:
# Save to Colab local storage
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"Model saved to: {OUTPUT_DIR}")

# Optional: save to Google Drive so it persists after Colab session ends
SAVE_TO_DRIVE = False   # set True to enable

if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    import shutil
    drive_path = "/content/drive/MyDrive/aragpt2-poetry"
    shutil.copytree(OUTPUT_DIR, drive_path, dirs_exist_ok=True)
    print(f"Also saved to Google Drive: {drive_path}")


## 10.  Generate Arabic Poetry

Try different prompts and decoding strategies.

**Decoding strategies:**
| Strategy | Character | Best for |
|---|---|---|
| `top_p` (nucleus) | Diverse, natural | Default — best balance |
| `beam` | Coherent, structured | Formal verse |
| `top_k` | Creative | Exploration |
| `greedy` | Deterministic | Debugging |


In [11]:
def generate_poetry(
    prompt: str,
    strategy: str = "top_p",
    max_new_tokens: int = 120,
    temperature: float = 0.9,
    top_p: float = 0.92,
    top_k: int = 50,
    num_beams: int = 5,
    num_outputs: int = 1,
) -> list[str]:
    """Generate Arabic poetry from a prompt."""
    inputs = tokenizer(prompt, return_tensors="pt").to(device)

    base_kwargs = {
        "max_new_tokens":       max_new_tokens,
        "num_return_sequences": num_outputs,
        "pad_token_id":         tokenizer.eos_token_id,
        "repetition_penalty":   1.3,
    }

    if strategy == "greedy":
        gen_kwargs = {**base_kwargs, "do_sample": False}
    elif strategy == "beam":
        gen_kwargs = {**base_kwargs, "do_sample": False,
                      "num_beams": num_beams, "early_stopping": True}
    elif strategy == "top_k":
        gen_kwargs = {**base_kwargs, "do_sample": True,
                      "temperature": temperature, "top_k": top_k}
    else:  # top_p
        gen_kwargs = {**base_kwargs, "do_sample": True,
                      "temperature": temperature, "top_p": top_p, "top_k": 0}

    model.eval()
    with torch.no_grad():
        outputs = model.generate(**inputs, **gen_kwargs)

    prompt_len = inputs["input_ids"].shape[1]
    results = []
    for out in outputs:
        text = tokenizer.decode(out[prompt_len:], skip_special_tokens=True)
        results.append(text)
    return results


In [12]:
# ── Generate from Al-Mutanabbi opening lines ─────────────────────────────────

prompts = [
    "الخيل والليل والبيداء",
    "على قدر أهل العزم",
    "أنا من أهوى ومن أهوى",
]

print("=" * 60)
print("  Generated Classical Arabic Poetry")
print("  Strategy: top_p (nucleus sampling)")
print("=" * 60)

for prompt in prompts:
    outputs = generate_poetry(prompt, strategy="top_p", max_new_tokens=100)
    print(f"\n── Prompt: {prompt}")
    print(f"{'─' * 50}")
    for text in outputs:
        print(prompt + " " + text)
    print()


  Generated Classical Arabic Poetry
  Strategy: top_p (nucleus sampling)

── Prompt: الخيل والليل والبيداء
──────────────────────────────────────────────────
الخيل والليل والبيداء 
تراهم بدجي امير المؤمنين علي البيت سماكهم
ولكن مثل عين الشمس تجتوي بها الاستناديد
اصبحت من العمر سبلاً واخلص نفسه لاغثني
وحياه وانطوي فؤادي وسقيلي بمسك
وطب جربي يا مولاي ما ادري ارمحي وجدي
وقد نزلت خفت في الاذناء والخمرري زادعي
تقضي ايضا عن الاقصي اخره الزايله الالهي
وحبيب ج


── Prompt: على قدر أهل العزم
──────────────────────────────────────────────────
على قدر أهل العزم 
محمد مقمر بدهم ال
عليه شان جاههم اهل المولي بفضلهم
كانوا له في زمان القرب طراوه اشرف العارفين حساما
وعينو اليوم قد ضاق الخناق صارفا
كم من نفر العاشقين يرنو بعين ال
نعم الله اكرم بيت العالمين سلطانا
اليوم يحسده فيه كل البعد مكرما
وامام الدين قطب المؤمنين ركنا مقفرايا موحدا خاضقيا بارزا مشيداورا نحيا توارياقادا
قد ذكرت فعلا


── Prompt: أنا من أهوى ومن أهوى
──────────────────────────────────────────────────
أنا من أهوى ومن أهوى 
حَيا حادي ا

In [ ]:
# ── Try your own prompt ───────────────────────────────────────────────────────

MY_PROMPT = "إذا غامرت في شرف"   # ← change this to any Arabic opening

print(f"Generating from: {MY_PROMPT!r}\n")

# Compare strategies side by side
for strategy in ["top_p", "beam", "greedy"]:
    result = generate_poetry(MY_PROMPT, strategy=strategy, max_new_tokens=80)
    print(f"[{strategy:8s}] {MY_PROMPT} {result[0]}")
    print()


In [14]:
# ── Generate multiple variations ─────────────────────────────────────────────

prompt = "الخيل والليل"
print(f"Generating 3 variations for: {prompt!r}\n")

variations = generate_poetry(
    prompt,
    strategy="top_p",
    temperature=1.1,    # higher temperature = more creative/random
    max_new_tokens=100,
    num_outputs=3,
)

for i, text in enumerate(variations, 1):
    print(f"── Variation {i} ──────────────────────────────────")
    print(f"{prompt} {text}")
    print()


Generating 3 variations for: 'الخيل والليل'

── Variation 1 ──────────────────────────────────
الخيل والليل  شدا
فتطل بالفن العدته والامال الحمر اوله الرجا الصياد
اكرم النجم يا عثمان حتي مكترث واهلاً وجوارها وتقطر
غادت عل والوجد اشطانها
قصه الجيره في اعيبه الجرح ويداهدداٌ وجدوها وصفوعا
هيقلت لها اسرار الغيب التي لا تلوي الظلال تتلقي التوي الكالك الاشجان والعنينا
يجني الاسير بلهيب وان لم يقر لي طيره مولته الاسفاشا موحدا�

── Variation 2 ──────────────────────────────────
الخيل والليل 
والجلد علي الظهر والعقل عزم خاطري
كما يريد حبيبي مكين من الباب الحديدي علا ووسقّا وطيرا
وسوي الهي يوم والحج ان يجف نارا ناريا
وجه الدمع والقلب يسيل دما وحنقا
طيبه بيض الوداع الطلقً وصباح الردود البالي تذكرهما
رفيق العود منه بات ينفهران للبعد مسرعا جدحاء
كثير غزيل البين يبكي عوسي
وهدي النهد جهرا ولي الليل يروي مر الوجد به مفزع
وم

── Variation 3 ──────────────────────────────────
الخيل والليل  يتضوع المرمحه
شاباك بشوله رعايه وعين هيفاء نعيمها
تبارك البكا عليك تعجبني المسك وهي محاله
ولعمري ترجع نفس النعيم رو

---
##  Summary

| Step | What happened |
|---|---|
| Dataset | Loaded `arbml/ashaar` — 180k+ classical Arabic poems |
| Preprocessing | Normalized Arabic text, filtered by length |
| Tokenizer | `aragpt2-base` SentencePiece (Arabic-native vocab) |
| Chunking | Poems concatenated with EOS, split into 256-token blocks |
| Model | `aragpt2-base` — 135M param Arabic GPT-2 |
| Training | AdamW + cosine LR + warmup + fp16 + gradient accumulation |
| Evaluation | Perplexity on held-out poems |
| Generation | top-p nucleus sampling with repetition penalty |

**References:**
- [AraGPT2 paper](https://arxiv.org/abs/2012.15520) — Antoun et al. (2020)
- [arbml/ashaar dataset](https://huggingface.co/datasets/arbml/ashaar)
- [GPT-2 paper](https://cdn.openai.com/better-language-models/language_models_are_unsupervised_multitask_learners.pdf) — Radford et al. (2019)
